In [1]:
from epyt import epanet
import pandas as pd
import numpy as np
import datetime
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point, LineString

In [104]:
# caminho ='Epanet Cavitacao\\NBN CND\\Representacao esquematica global Modelo Calibrado.inp'
# caminho = 'Epanet Gerados\\Recanto_Riacho2\\Reborn\\RCE RF2 Reborn.inp'
# caminho = 'Epanet Cavitacao\\Samambaia\\NOVAS VRPs 01.06.inp'
caminho = 'Epanet Cavitacao\\Gama 2\\GAM2 Calibrado.inp'

In [105]:
import chardet

def analisar_inp(caminho_arquivo):
    problemas = []

    # Detectar encoding do arquivo
    with open(caminho_arquivo, "rb") as f:
        raw = f.read()
        resultado = chardet.detect(raw)
        encoding_detectado = resultado['encoding']
        confianca = resultado['confidence']

    print(f"📌 Encoding detectado: {encoding_detectado} (confiança {confianca:.2f})")

    # Reabre com encoding detectado (ou força utf-8 se der erro)
    try:
        with open(caminho_arquivo, "r", encoding=encoding_detectado) as f:
            linhas = f.readlines()
    except:
        with open(caminho_arquivo, "r", encoding="utf-8", errors="replace") as f:
            linhas = f.readlines()

    # Percorre linha por linha
    for num, linha in enumerate(linhas, start=1):
        # Procura caracteres não ASCII
        for char in linha:
            if ord(char) > 127:  # fora do ASCII básico
                problemas.append((num, char, linha.strip()))
                break  # reporta só uma vez por linha

    if problemas:
        print("\n⚠️ Problemas encontrados (linhas com acentos/caracteres especiais):")
        for num, char, conteudo in problemas:
            print(f"  Linha {num}: caractere '{char}' → {conteudo}")
    else:
        print("\n✅ Nenhum caractere problemático encontrado.")

# Exemplo de uso
analisar_inp(caminho)


📌 Encoding detectado: ascii (confiança 1.00)

✅ Nenhum caractere problemático encontrado.


In [106]:
d = epanet(caminho)

EPANET version 20200 loaded (EPyT version v1.2.1 - Last Update: 09/01/2024).
Input File GAM2 Calibrado.inp loaded successfully.



# Metodo Simples de Nós para voltar o Gis

In [108]:
# ==========================
# FUNÇÃO PARA LER SEÇÃO (COM DESCRIÇÃO)
# ==========================
def ler_secao(inp, secao):
    dados = []
    captura = False

    with open(inp, 'r', encoding='utf-8') as f:
        for linha in f:
            linha = linha.strip()

            # Início da seção
            if linha.upper().startswith(f'[{secao.upper()}]'):
                captura = True
                continue

            if captura:
                # Fim da seção
                if linha.startswith('['):
                    break

                # Ignorar linhas vazias e cabeçalhos
                if not linha:
                    continue
                
                if linha.startswith(';'):
                    continue
                
                # 🔥 ignorar cabeçalho tipo "ID Elev Demand"
                if linha.upper().startswith('ID'):
                    continue

                # Separar comentário (descrição)
                if ";" in linha:
                    parte_dados, descricao = linha.split(";", 1)
                    descricao = descricao.strip()
                else:
                    parte_dados = linha
                    descricao = None

                # Separar campos principais
                partes = parte_dados.split()

                # Adicionar descrição no final
                partes.append(descricao)

                dados.append(partes)

    return dados


In [109]:
# ==========================
# COORDENADAS
# ==========================
coords = ler_secao(caminho, 'COORDINATES')

df_coords = pd.DataFrame(coords, columns=['ID', 'X', 'Y', 'DESC_TMP'])

# Converter tipos
df_coords['X'] = pd.to_numeric(df_coords['X'], errors='coerce')
df_coords['Y'] = pd.to_numeric(df_coords['Y'], errors='coerce')

# Remover coluna auxiliar
df_coords = df_coords[['ID', 'X', 'Y']]


# ==========================
# JUNCTIONS (NÓS)
# ==========================
junctions = ler_secao(caminho, 'JUNCTIONS')

dados = []

for linha in junctions:
    # Base: ID, Elevation, Demand, Pattern
    base = linha[:4]

    while len(base) < 4:
        base.append(None)

    # Descrição (último elemento)
    descricao = linha[-1] if len(linha) > 4 else None

    dados.append(base + [descricao])

df_nos = pd.DataFrame(
    dados,
    columns=['ID', 'ELEVATION', 'DEMAND', 'PATTERN', 'DESCRICAO']
)

# Converter tipos
df_nos['ELEVATION'] = pd.to_numeric(df_nos['ELEVATION'], errors='coerce')
df_nos['DEMAND'] = pd.to_numeric(df_nos['DEMAND'], errors='coerce')


# ==========================
# JUNÇÃO COM COORDENADAS
# ==========================
df_nos = df_nos.merge(df_coords, on='ID', how='left')


# ==========================
# GEO DATAFRAME
# ==========================
gdf_nos = gpd.GeoDataFrame(
    df_nos[['ID', 'ELEVATION', 'DEMAND', 'PATTERN', 'DESCRICAO']],
    geometry=gpd.points_from_xy(df_nos.X, df_nos.Y),
    crs='EPSG:31983'
)



In [110]:
df_nos

,ID,ELEVATION,DEMAND,PATTERN,DESCRICAO,X,Y
0,0,1098.87,5.025839,VZ1.DMC.GAM.003,,170226.069,8226404.331
1,1,1098.87,0.000000,VZ1.DMC.GAM.003,,170227.186,8226403.562
2,2,1088.07,0.004887,VZ1.DMC.GAM.004,,172360.184,8224124.542
3,3,1087.66,0.017747,VZ1.DMC.GAM.004,,172380.419,8224111.553
4,4,1088.41,0.000000,VZ1.DMC.GAM.004,,172336.941,8224137.667
...,...,...,...,...,...,...,...
7838,181,1168.17,0.000000,VZ1.SAT.GAM.031,,173686.614,8228052.361
7839,182,1176.32,0.000000,VZ1.SAT.GAM.031,,173356.000,8227917.812
7840,183,1155.41,0.000000,VZ1.SAT.GAM.031,,172647.683,8226876.515
7841,184,1161.20,0.000000,VZ1.SAT.GAM.031,,172509.855,8227276.654


In [92]:
df_nos['ID'].max()

'PM1.UDA.SAM.006'

In [93]:
df_nos['ID'].str.extract(r'(\d+)$')[0].dropna().astype(int).max()

21497

In [75]:
# ==========================
# EXPORTAR SHAPEFILE
# ==========================
gdf_nos.to_file('Shape\\Vicente\\nos_epanet VCP Reborn.shp')

# Metodo de Nós com pressão 

In [111]:
import datetime
import pandas as pd

d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()

tstep = 1

P = []
horarios = []

proxima_hora = 3600  # primeira hora a capturar

while tstep > 0:

    t = d.runHydraulicAnalysis()

    pressao_no_momento = d.getNodePressure()

    # ======================================================
    # Captura apenas quando ultrapassar a próxima hora alvo
    # ======================================================
    if t >= proxima_hora:

        P.append(pressao_no_momento)

        horarios.append(str(datetime.timedelta(seconds=proxima_hora)))

        proxima_hora += 3600  # avança 1 hora

    tstep = d.nextHydraulicAnalysisStep()

d.closeHydraulicAnalysis()

In [112]:
ids_nos = d.getNodeNameID()
df_pressao = pd.DataFrame(P).T
df_pressao.columns = [f'P_{str(i).zfill(2)}' for i in range(df_pressao.shape[1])]
df_pressao['ID'] = ids_nos

cols = ['ID'] + [col for col in df_pressao.columns if col != 'ID']
df_pressao = df_pressao[cols]

In [113]:
def ler_secao(inp, secao):
    dados = []
    captura = False

    with open(inp, 'r', encoding='utf-8') as f:
        for linha in f:
            linha = linha.strip()

            if linha.startswith(f'[{secao}]'):
                captura = True
                continue

            if captura:
                if linha.startswith('['):
                    break
                if linha and not linha.startswith(';'):
                    dados.append(linha.split())

    return dados

In [114]:
coords = ler_secao(caminho, 'COORDINATES')

df_coords = pd.DataFrame(coords, columns=['ID', 'X', 'Y'])
df_coords['X'] = df_coords['X'].astype(float)
df_coords['Y'] = df_coords['Y'].astype(float)

In [115]:
junctions = ler_secao(caminho, 'JUNCTIONS')

dados = []

for linha in junctions:
    linha = linha[:4]  # pega no máximo 4 campos

    while len(linha) < 4:
        linha.append(None)

    dados.append(linha)

df_nos = pd.DataFrame(
    dados,
    columns=['ID', 'ELEVATION', 'DEMAND', 'PATTERN']
)

df_nos['ELEVATION'] = pd.to_numeric(df_nos['ELEVATION'], errors='coerce')
df_nos['DEMAND'] = pd.to_numeric(df_nos['DEMAND'], errors='coerce')

df_nos = df_nos.merge(df_coords, on='ID', how='left')

In [116]:
gdf_nos = gpd.GeoDataFrame(
    df_nos,
    geometry=gpd.points_from_xy(df_nos.X, df_nos.Y),
    crs='EPSG:31983'
)

In [117]:
gdf_nos = gdf_nos.merge(df_pressao, on='ID', how='left')

In [101]:
gdf_nos['P_00'].notna().sum()

21465

In [118]:
gdf_nos

,ID,ELEVATION,DEMAND,PATTERN,X,Y,geometry,P_00,P_01,P_02,...,P_14,P_15,P_16,P_17,P_18,P_19,P_20,P_21,P_22,P_23
0,0,1098.87,5.025839,VZ1.DMC.GAM.003,170226.069,8226404.331,POINT (170226.069 8226404.331),27.203938,27.285402,27.332495,...,30.932829,30.999498,30.886606,30.859112,30.786030,30.869478,31.218443,31.352964,26.726601,26.914049
1,1,1098.87,0.000000,VZ1.DMC.GAM.003,170227.186,8226403.562,POINT (170227.186 8226403.562),27.203955,27.285416,27.332506,...,30.932896,30.999561,30.886671,30.859179,30.786100,30.869545,31.218498,31.353014,26.726633,26.914076
2,2,1088.07,0.004887,VZ1.DMC.GAM.004,172360.184,8224124.542,POINT (172360.184 8224124.542),28.060034,27.936127,27.926256,...,26.482719,26.308331,26.287571,26.103323,26.347210,26.704393,27.141718,27.192022,27.625965,27.967676
3,3,1087.66,0.017747,VZ1.DMC.GAM.004,172380.419,8224111.553,POINT (172380.419 8224111.553),28.468431,28.344875,28.335007,...,26.884109,26.709587,26.688280,26.504171,26.747919,27.106184,27.545010,27.597101,28.032658,28.375402
4,4,1088.41,0.000000,VZ1.DMC.GAM.004,172336.941,8224137.667,POINT (172336.941 8224137.667),27.721851,27.597548,27.587677,...,26.152485,25.978251,25.958109,25.773705,26.017750,26.373705,26.809326,26.857607,27.289717,27.630257
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7838,181,1168.17,0.000000,VZ1.SAT.GAM.031,173686.614,8228052.361,POINT (173686.614 8228052.361),21.690763,21.895899,21.952902,...,19.818613,19.748638,19.897657,19.828646,19.739519,19.956314,20.244165,20.515673,20.994345,21.411591
7839,182,1176.32,0.000000,VZ1.SAT.GAM.031,173356.000,8227917.812,POINT (173356 8227917.812),32.380020,33.020981,33.264320,...,27.062195,27.185696,27.319317,27.186049,26.795149,27.358995,28.156918,29.185497,30.369995,31.424654
7840,183,1155.41,0.000000,VZ1.SAT.GAM.031,172647.683,8226876.515,POINT (172647.683 8226876.515),51.687668,52.573605,52.932491,...,43.888786,44.012062,44.210594,44.012703,43.490612,44.315128,45.428169,46.935623,48.675243,50.282894
7841,184,1161.20,0.000000,VZ1.SAT.GAM.031,172509.855,8227276.654,POINT (172509.855 8227276.654),46.100296,46.955257,47.299538,...,38.615120,38.738449,38.928734,38.739033,38.233570,39.025112,40.098320,41.545185,43.214642,44.752384


In [119]:
gdf_nos.to_file('Shape\\Gama 2\\nos_pressao_24h GAM2.shp')

In [46]:
# gdf_nos

# Rede

In [76]:
import pandas as pd
import datetime

d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()

tstep = 1

Q = []
V = []
HL = []

while tstep > 0:
    t = d.runHydraulicAnalysis()

    vazao = d.getLinkFlows()
    velocidade = d.getLinkVelocity()
    headloss = d.getLinkHeadloss()

    Q.append(vazao)
    V.append(velocidade)
    HL.append(headloss)

    tstep = d.nextHydraulicAnalysisStep()

d.closeHydraulicAnalysis()

In [ ]:
ids_links = d.getLinkNameID()

df_q = pd.DataFrame(Q).T
df_q.columns = [f'Q_{str(i).zfill(2)}' for i in range(df_q.shape[1])]
df_q['ID'] = ids_links

df_v = pd.DataFrame(V).T
df_v.columns = [f'V_{str(i).zfill(2)}' for i in range(df_v.shape[1])]
df_v['ID'] = ids_links

df_v = pd.DataFrame(V).T
df_v.columns = [f'V_{str(i).zfill(2)}' for i in range(df_v.shape[1])]
df_v['ID'] = ids_links

df_hl = pd.DataFrame(HL).T
df_hl.columns = [f'HL_{str(i).zfill(2)}' for i in range(df_hl.shape[1])]
df_hl['ID'] = ids_links

# gdf_rede['ID'] = gdf_rede['ID'].astype(str)
df_q['ID'] = df_q['ID'].astype(str)
df_v['ID'] = df_v['ID'].astype(str)
df_hl['ID'] = df_hl['ID'].astype(str)

In [77]:
pipes = ler_secao(caminho, 'PIPES')

dados_pipes = []

for linha in pipes:
    linha = linha[:8]

    while len(linha) < 8:
        linha.append(None)

    dados_pipes.append(linha)

df_rede = pd.DataFrame(
    dados_pipes,
    columns=[
        'ID', 'NO1', 'NO2',
        'LENGTH', 'DIAMETER',
        'ROUGHNESS', 'MINORLOSS', 'STATUS'
    ]
)

for col in ['LENGTH', 'DIAMETER', 'ROUGHNESS', 'MINORLOSS']:
    df_rede[col] = pd.to_numeric(df_rede[col], errors='coerce')

In [78]:
from shapely.geometry import LineString

coord_dict = df_coords.set_index('ID')[['X', 'Y']].to_dict('index')

linhas = []

for _, row in df_rede.iterrows():
    no1 = row['NO1']
    no2 = row['NO2']

    if no1 in coord_dict and no2 in coord_dict:
        linha = LineString([
            (coord_dict[no1]['X'], coord_dict[no1]['Y']),
            (coord_dict[no2]['X'], coord_dict[no2]['Y'])
        ])

        linhas.append({
            **row,
            'geometry': linha
        })

gdf_rede = gpd.GeoDataFrame(linhas, crs='EPSG:31983')

In [574]:
df_links = df_q.merge(df_v, on='ID').merge(df_hl, on='ID')

In [575]:
gdf_rede['ID'] = gdf_rede['ID'].astype(str)
df_links['ID'] = df_links['ID'].astype(str)

gdf_rede = gdf_rede.merge(df_links, on='ID', how='left')

In [79]:
gdf_rede

,ID,NO1,NO2,LENGTH,DIAMETER,ROUGHNESS,MINORLOSS,STATUS,geometry
0,0,11075,11076,45.3940,63.0,140.0,0.0,Open,"LINESTRING (177083.73 8250841.905, 177083.742 ..."
1,1,11073,11074,107.2596,63.0,140.0,0.0,Open,"LINESTRING (176976.546 8250834.268, 177083.803..."
2,2,11074,11075,6.8934,63.0,140.0,0.0,Open,"LINESTRING (177083.803 8250835.012, 177083.73 ..."
3,3,11076,11077,5.8388,63.0,140.0,0.0,Open,"LINESTRING (177083.742 8250887.299, 177083.938..."
4,4,11077,11078,95.1805,63.0,140.0,0.0,Open,"LINESTRING (177083.938 8250893.135, 176988.759..."
...,...,...,...,...,...,...,...,...,...
12386,12378,11167,11169,1000.0000,12.0,100.0,0.0,Open,"LINESTRING (173913.498 8248826.083, 173953.941..."
12387,12380,11174,11175,1000.0000,12.0,100.0,0.0,Open,"LINESTRING (173940.999 8248880.277, 173906.218..."
12388,12381,11175,11166,1000.0000,12.0,100.0,0.0,Open,"LINESTRING (173906.218 8248845.496, 173810.773..."
12389,12382,11166,11170,1000.0000,12.0,100.0,0.0,Open,"LINESTRING (173810.773 8248926.382, 173739.593..."


In [80]:
gdf_rede.to_file('Shape\\Vicente\\rede VCP Reborn.shp')

# VRP

In [578]:
valves = ler_secao(caminho, 'VALVES')

In [579]:
dados_vrp = []

for linha in valves:
    linha = linha[:7]

    while len(linha) < 7:
        linha.append(None)

    dados_vrp.append(linha)

df_vrp = pd.DataFrame(
    dados_vrp,
    columns=[
        'ID', 'NO1', 'NO2',
        'DIAMETER', 'TYPE',
        'SETTING', 'MINORLOSS'
    ]
)

df_vrp = df_vrp[df_vrp['TYPE'] == 'PRV']

In [580]:
from shapely.geometry import Point

vrps = []

for _, row in df_vrp.iterrows():
    no1 = row['NO1']

    if no1 in coord_dict:
        vrps.append({
            **row,
            'geometry': Point(
                coord_dict[no1]['X'],
                coord_dict[no1]['Y']
            )
        })

gdf_vrp = gpd.GeoDataFrame(vrps, crs='EPSG:31983')

In [581]:
gdf_vrp.to_file('Shape\\NBN CND\\calibrado\\vrp.shp')

In [582]:
lista = d.getLinkValveNameID()

# Ordenar em ordem alfabética
lista_ordenada = sorted(lista)

# Printar cada item
for item in lista_ordenada:
    print(f"'{item}',")


'VRP.CND.001',
'VRP.CND.002',
'VRP.CND.003',
'VRP.CND.004',
'VRP.GUA.014',
'VRP.NBN.001',
'VRP.NBN.002',
'VRP.NBN.003',
'VRP.NBN.005',
'VRP.SPW.010',


In [583]:
lista_valvulas = ['VRP.CND.001',
'VRP.CND.002',
'VRP.CND.003',
'VRP.CND.004',
'VRP.GUA.014',
'VRP.NBN.001',
'VRP.NBN.002',
'VRP.NBN.003',
'VRP.NBN.005',
'VRP.SPW.010',]

In [584]:
resultado_vrps = {}

In [585]:
dict_link_name_index = {
    nome: idx for nome, idx in zip(d.getLinkNameID(), d.getLinkIndex())
}

for vrp in lista_valvulas:

    print(f'Processando {vrp}')

    if vrp not in dict_link_name_index:
        continue

    idx_link = dict_link_name_index[vrp]

    status_original = d.getLinkInitialStatus()[idx_link - 1]

    # fecha somente essa VRP
    d.setLinkInitialStatus(idx_link, 0)

    d.openHydraulicAnalysis()
    d.initializeHydraulicAnalysis()

    tstep = 1
    pressao_final = None

    while tstep > 0:
        d.runHydraulicAnalysis()
        pressao_final = d.getNodePressure()
        tstep = d.nextHydraulicAnalysisStep()

    d.closeHydraulicAnalysis()

    # pressão baixa
    zeros = np.where(np.array(pressao_final) <= 1)[0]

    ids_zero = [d.getNodeNameID()[i] for i in zeros]

    resultado_vrps[vrp] = ids_zero

    # restaura
    d.setLinkInitialStatus(idx_link, status_original)

Processando VRP.CND.001
Processando VRP.CND.002
Processando VRP.CND.003
Processando VRP.CND.004
Processando VRP.GUA.014
Processando VRP.NBN.001
Processando VRP.NBN.002
Processando VRP.NBN.003
Processando VRP.NBN.005
Processando VRP.SPW.010


In [586]:
tamanho_vrp = {
    vrp: len(nos)
    for vrp, nos in resultado_vrps.items()
}

In [587]:
df_nos

,ID,ELEVATION,DEMAND,PATTERN,X,Y
0,0,1026.20,0.093873,VZ1.SAT.NBN.014,184364.028,8245057.145
1,1,1026.20,0.077392,VZ1.SAT.NBN.014,184365.120,8245055.470
2,2,1026.20,0.038697,VZ1.SAT.NBN.014,184365.848,8245054.355
3,3,1035.92,0.000000,VZ1.SAT.NBN.014,184122.502,8245560.306
4,4,1036.34,0.000000,VZ1.SAT.NBN.014,184125.377,8245557.215
...,...,...,...,...,...,...
2856,2514,1049.70,0.180000,;,179939.491,8245420.117
2857,2515,1049.70,0.000000,;,179949.168,8245435.003
2858,2516,1049.70,0.000000,;,179961.821,8245423.838
2859,2517,1049.70,0.000000,;,179952.889,8245410.441


In [588]:
df_nos = pd.DataFrame({
    'NodeID': d.getNodeNameID()
})

df_nos['ZP'] = ''

In [589]:
for node in df_nos['NodeID']:

    vrps_afetaram = []

    for vrp, nos in resultado_vrps.items():
        if node in nos:
            vrps_afetaram.append(vrp)

    if vrps_afetaram:

        vrp_escolhida = min(
            vrps_afetaram,
            key=lambda x: tamanho_vrp[x]
        )

        df_nos.loc[df_nos['NodeID'] == node, 'ZP'] = vrp_escolhida

## Dois metodos ZP

In [622]:
lista = d.getLinkValveNameID()

# Ordenar em ordem alfabética
lista_ordenada = sorted(lista)

# Printar cada item
for item in lista_ordenada:
    print(f"'{item}',")

'VRP.CND.001',
'VRP.CND.002',
'VRP.CND.003',
'VRP.CND.004',
'VRP.GUA.014',
'VRP.NBN.001',
'VRP.NBN.002',
'VRP.NBN.003',
'VRP.NBN.005',
'VRP.SPW.010',


In [623]:
lista_valvulas = ['VRP.CND.001',
'VRP.CND.002',
'VRP.CND.003',
'VRP.CND.004',
'VRP.GUA.014',
'VRP.NBN.001',
'VRP.NBN.002',
'VRP.NBN.003',
'VRP.NBN.005',
'VRP.SPW.010']

In [624]:
def obter_nos_da_valvula(nome_valvula):

    dict_valve_name_index = {
        nome: index
        for index, nome in enumerate(d.getLinkValveNameID(), start=1)
    }

    valve_id = d.getLinkValveNameID(
        [dict_valve_name_index[nome_valvula]]
    )

    nos_conectados = d.getNodesConnectingLinksID(valve_id)

    if isinstance(nos_conectados, np.ndarray) and nos_conectados.ndim > 1:
        nos_conectados = nos_conectados[0]

    return [str(no) for no in nos_conectados]

In [593]:
dict_link_name_index = {
    nome: idx for nome, idx in zip(d.getLinkNameID(), d.getLinkIndex())
}

for vrp in lista_valvulas:

    print(f'Processando {vrp}')

    if vrp not in dict_link_name_index:
        continue

    idx_link = dict_link_name_index[vrp]

    status_original = d.getLinkInitialStatus()[idx_link - 1]

    # fecha somente essa VRP
    d.setLinkInitialStatus(idx_link, 0)

    d.openHydraulicAnalysis()
    d.initializeHydraulicAnalysis()

    tstep = 1
    pressao_final = None

    while tstep > 0:
        d.runHydraulicAnalysis()
        pressao_final = d.getNodePressure()
        tstep = d.nextHydraulicAnalysisStep()

    d.closeHydraulicAnalysis()

    # pressão baixa
    zeros = np.where(np.array(pressao_final) <= 1)[0]

    ids_zero = [d.getNodeNameID()[i] for i in zeros]

    resultado_vrps[vrp] = ids_zero

    # restaura
    d.setLinkInitialStatus(idx_link, status_original)

Processando VRP.CND.001
Processando VRP.CND.002
Processando VRP.CND.003
Processando VRP.CND.004
Processando VRP.GUA.014
Processando VRP.NBN.001
Processando VRP.NBN.002
Processando VRP.NBN.003
Processando VRP.NBN.005
Processando VRP.NBN.NOVA.001
Processando VRP.NBN.NOVA.002
Processando VRP.NBN.NOVA.003
Processando VRP.NBN.NOVA.004
Processando VRP.NBN.NOVA.005
Processando VRP.NBN.NOVA.006
Processando VRP.NBN.NOVA.007
Processando VRP.NBN.NOVA.008
Processando VRP.NBN.NOVA.009
Processando VRP.NBN.NOVA.010
Processando VRP.NBN.NOVA.011
Processando VRP.SPW.010
Processando VRP.SPW.NOVA.001
Processando VRP.SPW.NOVA.002
Processando VRP.SPW.NOVA.003


In [594]:
tamanho_vrp = {
    vrp: len(nos)
    for vrp, nos in resultado_vrps.items()
}

In [595]:
df_nos = pd.DataFrame({
    'NodeID': d.getNodeNameID()
})

df_nos['ZP_zeradas'] = ''

In [596]:
for node in df_nos['NodeID']:

    vrps_afetaram = []

    for vrp, nos in resultado_vrps.items():
        if node in nos:
            vrps_afetaram.append(vrp)

    if vrps_afetaram:

        vrp_escolhida = min(
            vrps_afetaram,
            key=lambda x: tamanho_vrp[x]
        )

        df_nos.loc[df_nos['NodeID'] == node, 'ZP_zeradas'] = vrp_escolhida

In [597]:
df_nos['ZP_zeradas'].value_counts()

ZP_zeradas
               2861
VRP.CND.001       1
Name: count, dtype: int64

In [449]:
df_nos['ZP_zerado'].value_counts()

ZP_zerado
                    3464
VRP.SSB.012A        1578
VRP_FICTICIA_EBO     559
VRP.SSB.015          503
VRP_FICTICIA1        317
VRP_FICTICIA         180
VRP.SSB.NOVA.001     179
VRP_FICTICIA2        178
VRP.SSB.004           73
VRP.SSB.NOVA.002      45
VRP.SSB.002           12
Name: count, dtype: int64

In [450]:
df_nos['ZP_reducao'].value_counts()

ZP_reducao
                    3468
VRP.SSB.012A        1580
VRP_FICTICIA_EBO     559
VRP.SSB.015          509
VRP_FICTICIA1        317
VRP_FICTICIA         180
VRP.SSB.NOVA.001     179
VRP_FICTICIA2        178
VRP.SSB.004           73
VRP.SSB.NOVA.002      45
Name: count, dtype: int64

## Aparentemente deu bom (para zerado e redução)

In [625]:
import numpy as np
import pandas as pd
import warnings

# -----------------------------------------------------------
# 0. CONFIGURAÇÕES E SILENCIAMENTO DE AVISOS
# -----------------------------------------------------------
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# -----------------------------------------------------------
# 1. FUNÇÕES DE SUPORTE (TOPOLOGIA E CAPTURA)
# -----------------------------------------------------------

def obter_nos_da_valvula(nome_valvula):
    """Retorna os IDs dos dois nós conectados à válvula de forma 'achatada'."""
    try:
        dict_valve_name_index = {nome: index for index, nome in enumerate(d.getLinkValveNameID(), start=1)}
        if nome_valvula not in dict_valve_name_index:
            return []
        v_idx = dict_valve_name_index[nome_valvula]
        valve_id = d.getLinkValveNameID([v_idx])
        nos_raw = d.getNodesConnectingLinksID(valve_id)
        nos_flat = np.array(nos_raw).flatten()
        return [str(no) for no in nos_flat if str(no) != '']
    except:
        return []

def testar_cenario(links_para_fechar):
    """
    Recebe uma lista de tuplas (id_link, index_link), fecha todos,
    roda a hidráulica e retorna os nós impactados.
    """
    status_originais = []
    for l_id, l_idx in links_para_fechar:
        status_originais.append((l_idx, d.getLinkInitialStatus()[l_idx-1]))
        d.setLinkInitialStatus(l_idx, 0)
    
    z_set, r_set = set(), set()
    try:
        d.openHydraulicAnalysis()
        d.initializeHydraulicAnalysis()
        d.runHydraulicAnalysis()
        p_pos = np.array(d.getNodePressure())
        d.closeHydraulicAnalysis()
        
        z_idx = np.where(p_pos <= 1.0)[0]
        z_set = set([str(d.getNodeNameID()[i]) for i in z_idx])
        
        queda = pressao_base - p_pos
        r_idx = np.where(queda >= 10.0)[0]
        r_set = set([str(d.getNodeNameID()[i]) for i in r_idx]).union(z_set)
    except:
        if d.getHydraulicAnalysisOpen(): d.closeHydraulicAnalysis()
    finally:
        for l_idx, status in status_originais:
            d.setLinkInitialStatus(l_idx, status)
            
    return z_set, r_set

# -----------------------------------------------------------
# 2. MAPEAMENTO INICIAL E SIMULAÇÃO BASE
# -----------------------------------------------------------

dict_node_name_index = {str(nome): idx for nome, idx in zip(d.getNodeNameID(), d.getNodeIndex())}
dict_link_name_index = {str(nome): idx for nome, idx in zip(d.getLinkNameID(), d.getLinkIndex())}

print("🚀 Rodando simulação base...")
d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()
d.runHydraulicAnalysis()
pressao_base = np.array(d.getNodePressure())
d.closeHydraulicAnalysis()

df_link_topo = pd.DataFrame({
    "id": [str(x) for x in d.getLinkNameID()],
    "index": d.getLinkIndex(),
    "startnode": [str(d.getNodeNameID()[ids[0]-1]) for ids in d.getLinkNodesIndex()],
    "endnode": [str(d.getNodeNameID()[ids[1]-1]) for ids in d.getLinkNodesIndex()]
})

# -----------------------------------------------------------
# 3. LOOP EXAUSTIVO DE CENÁRIOS CONCORRENTES
# -----------------------------------------------------------

resultado_zerado_bruto = {}
resultado_reducao_bruto = {}

print(f"⚙️ Analisando {len(lista_valvulas)} VRPs com Força Bruta Adjacente...")

for vrp in lista_valvulas:
    print(f'VRP: {vrp}', end=" | ")
    nos_v = obter_nos_da_valvula(vrp)
    if len(nos_v) < 2:
        print("⚠️ Erro nos nós.")
        continue
    
    p1 = pressao_base[dict_node_name_index[nos_v[0]]-1]
    p2 = pressao_base[dict_node_name_index[nos_v[1]]-1]
    no_jusante = nos_v[0] if p1 < p2 else nos_v[1]

    cenarios = {
        "VRP_SOH": [(vrp, dict_link_name_index[vrp])] if vrp in dict_link_name_index else [],
        "LADO_A": [],
        "LADO_B": [],
        "JUSANTE_TOTAL": []
    }

    for i, no in enumerate(nos_v):
        cand = df_link_topo[((df_link_topo['startnode'] == no) | (df_link_topo['endnode'] == no)) & 
                            (df_link_topo['id'] != vrp) & (~df_link_topo['id'].str.contains('VRP'))]
        if not cand.empty:
            cenarios[f"LADO_{'A' if i == 0 else 'B'}"] = [(cand.iloc[0]['id'], cand.iloc[0]['index'])]

    saidas = df_link_topo[((df_link_topo['startnode'] == no_jusante) | (df_link_topo['endnode'] == no_jusante)) & (df_link_topo['id'] != vrp)]
    if not saidas.empty:
        cenarios["JUSANTE_TOTAL"] = list(zip(saidas['id'], saidas['index']))

    melhor_z, melhor_r = set(), set()
    for nome_cenario, links in cenarios.items():
        if not links: continue
        z, r = testar_cenario(links)
        if len(r) > len(melhor_r):
            melhor_z, melhor_r = z, r

    resultado_zerado_bruto[vrp] = melhor_z
    resultado_reducao_bruto[vrp] = melhor_r
    print(f"✅ Z: {len(melhor_z)} / R: {len(melhor_r)}")

# -----------------------------------------------------------
# 4. ATRIBUIÇÃO EXCLUSIVA (REGRA DA MENOR MANCHA)
# -----------------------------------------------------------

print("\n🎯 Setorizando nós e gerando df_ZP...")

tamanhos_z = {v: len(nos) for v, nos in resultado_zerado_bruto.items()}
tamanhos_r = {v: len(nos) for v, nos in resultado_reducao_bruto.items()}

atribuicao_z, atribuicao_r = {}, {}
todos_os_nos = [str(n) for n in d.getNodeNameID()]

for node in todos_os_nos:
    cz = [v for v, nos in resultado_zerado_bruto.items() if node in nos]
    if cz: atribuicao_z[node] = min(cz, key=lambda x: tamanhos_z[x])
    
    cr = [v for v, nos in resultado_reducao_bruto.items() if node in nos]
    if cr: atribuicao_r[node] = min(cr, key=lambda x: tamanhos_r[x])

# -----------------------------------------------------------
# 5. CONSOLIDAÇÃO NO DF_ZP E RELATÓRIO FINAL
# -----------------------------------------------------------

df_ZP = pd.DataFrame({'NodeID': todos_os_nos})
df_ZP['ZP_zerado'] = df_ZP['NodeID'].map(atribuicao_z).fillna('')
df_ZP['ZP_reducao'] = df_ZP['NodeID'].map(atribuicao_r).fillna('')

# Contagem Exclusiva para o Comparativo
count_z = df_ZP[df_ZP['ZP_zerado'] != '']['ZP_zerado'].value_counts()
count_r = df_ZP[df_ZP['ZP_reducao'] != '']['ZP_reducao'].value_counts()

print("\n" + "="*40)
print(f"📊 RESUMO DO MAPA df_ZP")
print(f"SOMA TOTAL NÓS ZERADOS: {count_z.sum()}")
print(f"SOMA TOTAL NÓS REDUÇÃO: {count_r.sum()}")
print(f"TOTAL DE NÓS EXISTENTES: {len(df_ZP)}")
print("="*40)

comparativo = pd.DataFrame({'VRP': lista_valvulas})
comparativo['Nos_Zerado'] = comparativo['VRP'].map(count_z).fillna(0).astype(int)
comparativo['Nos_Reducao'] = comparativo['VRP'].map(count_r).fillna(0).astype(int)
comparativo = comparativo.sort_values(by='VRP').reset_index(drop=True)

# Exibição do comparativo final
comparativo

🚀 Rodando simulação base...
⚙️ Analisando 10 VRPs com Força Bruta Adjacente...
VRP: VRP.CND.001 | ✅ Z: 859 / R: 859
VRP: VRP.CND.002 | ✅ Z: 119 / R: 119
VRP: VRP.CND.003 | ✅ Z: 126 / R: 126
VRP: VRP.CND.004 | ✅ Z: 123 / R: 123
VRP: VRP.GUA.014 | ✅ Z: 181 / R: 181
VRP: VRP.NBN.001 | ✅ Z: 118 / R: 118
VRP: VRP.NBN.002 | ✅ Z: 90 / R: 90
VRP: VRP.NBN.003 | ✅ Z: 78 / R: 78
VRP: VRP.NBN.005 | ✅ Z: 206 / R: 206
VRP: VRP.SPW.010 | ✅ Z: 2854 / R: 2854

🎯 Setorizando nós e gerando df_ZP...

📊 RESUMO DO MAPA df_ZP
SOMA TOTAL NÓS ZERADOS: 2854
SOMA TOTAL NÓS REDUÇÃO: 2854
TOTAL DE NÓS EXISTENTES: 2862


,VRP,Nos_Zerado,Nos_Reducao
0,VRP.CND.001,493,493
1,VRP.CND.002,118,118
2,VRP.CND.003,125,125
3,VRP.CND.004,122,122
4,VRP.GUA.014,180,180
5,VRP.NBN.001,117,117
6,VRP.NBN.002,89,89
7,VRP.NBN.003,78,78
8,VRP.NBN.005,205,205
9,VRP.SPW.010,1327,1327


In [626]:
df_ZP

,NodeID,ZP_zerado,ZP_reducao
0,0,VRP.CND.002,VRP.CND.002
1,1,VRP.CND.002,VRP.CND.002
2,2,VRP.CND.002,VRP.CND.002
3,3,VRP.CND.003,VRP.CND.003
4,4,VRP.CND.003,VRP.CND.003
...,...,...,...
2857,2515,VRP.SPW.010,VRP.SPW.010
2858,2516,VRP.SPW.010,VRP.SPW.010
2859,2517,VRP.SPW.010,VRP.SPW.010
2860,379,VRP.SPW.010,VRP.SPW.010


In [627]:
# Unindo o df_ZP ao gdf_nos
# Usamos left_on e right_on porque os nomes das colunas de identificação são diferentes
gdf_nos = gdf_nos.merge(
    df_ZP, 
    left_on='ID', 
    right_on='NodeID', 
    how='left'
)

# Opcional: Remover a coluna 'NodeID' que ficará duplicada após o merge
gdf_nos = gdf_nos.drop(columns=['NodeID'])

# Verificar se os dados foram adicionados
# print(gdf_nos[['ID', 'ZP_zerado', 'ZP_reducao']].head())

In [628]:
gdf_nos.to_file('Shape\\NBN CND\\calibrado\\nos_pressao_24h regulagem.shp')

## Corrigindo

In [528]:
import numpy as np
import pandas as pd
import datetime
import warnings

# ---------------------------
# 0. CONFIGURAÇÕES INICIAIS
# ---------------------------
# Silenciar os avisos de pressão negativa para o console não ficar poluído
warnings.filterwarnings('ignore', category=UserWarning, message='.*negative pressures.*')

# ---------------------------
# 1. FUNÇÕES DE APOIO (LÓGICA LENDÁRIA)
# ---------------------------
def obter_nos_da_valvula(nome_valvula):
    """Retorna os IDs dos dois nós conectados à válvula."""
    try:
        # Pega o índice da válvula (base 1)
        dict_valve_name_index = {nome: idx for idx, nome in enumerate(d.getLinkValveNameID(), start=1)}
        if nome_valvula not in dict_valve_name_index:
            return None
        
        v_idx = dict_valve_name_index[nome_valvula]
        # Pega o Link ID associado à essa válvula
        link_id = d.getLinkValveNameID([v_idx])[0]
        nos_conectados = d.getNodesConnectingLinksID(link_id)
        
        if isinstance(nos_conectados, np.ndarray) and nos_conectados.ndim > 1:
            nos_conectados = nos_conectados[0]
        return [str(no) for no in nos_conectados]
    except:
        return None

# ---------------------------
# 2. MAPEAMENTO DA REDE E SIMULAÇÃO BASE
# ---------------------------
# Dicionários de suporte
dict_node_name_index = {nome: idx for nome, idx in zip(d.getNodeNameID(), d.getNodeIndex())}
dict_link_name_index = {nome: idx for nome, idx in zip(d.getLinkNameID(), d.getLinkIndex())}

print("🚀 Rodando simulação base para identificar Montante/Jusante...")
d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()
tstep = 1
pressao_base = None
while tstep > 0:
    d.runHydraulicAnalysis()
    pressao_base = np.array(d.getNodePressure())
    tstep = d.nextHydraulicAnalysisStep()
d.closeHydraulicAnalysis()

# Criar DataFrame de links robusto (Lógica Lendária)
dict_ids = {index: nome for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}
startnode = [dict_ids[ids[0]] for ids in d.getLinkNodesIndex()]
endnode = [dict_ids[ids[1]] for ids in d.getLinkNodesIndex()]
df_link_topo = pd.DataFrame({
    "id": d.getLinkNameID(),
    "startnode": startnode,
    "endnode": endnode,
    "diameter": d.getLinkDiameter()
})

# ---------------------------
# 3. LOOP PRINCIPAL (ZERADOS + REDUÇÃO)
# ---------------------------
resultado_zerado = {}
resultado_reducao = {}
limite_queda = 15

for vrp in lista_valvulas:
    print(f'➡️ Processando VRP: {vrp}')
    
    # Identifica nós montante/jusante pela pressão (Lógica Robusta)
    nos = obter_nos_da_valvula(vrp)
    if not nos:
        print(f"  ⚠️ VRP {vrp} não encontrada na rede.")
        continue
    
    # Nó com maior pressão na base é o Montante
    idx0 = dict_node_name_index[nos[0]] - 1
    idx1 = dict_node_name_index[nos[1]] - 1
    
    if pressao_base[idx0] > pressao_base[idx1]:
        no_montante, no_jusante = nos[0], nos[1]
    else:
        no_montante, no_jusante = nos[1], nos[0]
    
    print(f"  🔗 Montante: {no_montante} | Jusante: {no_jusante}")

    # Pega o índice do link para fechar
    if vrp not in dict_link_name_index:
        continue
    
    idx_link = dict_link_name_index[vrp]
    status_original = d.getLinkInitialStatus()[idx_link - 1]

    # FECHA A VRP
    d.setLinkInitialStatus(idx_link, 0)

    try:
        d.openHydraulicAnalysis()
        d.initializeHydraulicAnalysis()
        tstep = 1
        pressao_vrp_fechada = None
        while tstep > 0:
            d.runHydraulicAnalysis()
            pressao_vrp_fechada = np.array(d.getNodePressure())
            tstep = d.nextHydraulicAnalysisStep()
        d.closeHydraulicAnalysis()

        # --- ANÁLISE 1: ZERADOS ---
        zeros_idx = np.where(pressao_vrp_fechada <= 1)[0]
        resultado_zerado[vrp] = [d.getNodeNameID()[i] for i in zeros_idx]

        # --- ANÁLISE 2: REDUÇÃO ---
        queda = pressao_base - pressao_vrp_fechada
        reducao_idx = np.where(queda >= limite_queda)[0]
        resultado_reducao[vrp] = [d.getNodeNameID()[i] for i in reducao_idx]
        
        print(f"  ✅ Zerados: {len(resultado_zerado[vrp])} | Redução: {len(resultado_reducao[vrp])}")

    finally:
        # Restaura sempre o status original
        d.setLinkInitialStatus(idx_link, status_original)

# ---------------------------
# 4. CONSOLIDAÇÃO NO DF_NOS
# ---------------------------
tamanho_zerado = {vrp: len(nos) for vrp, nos in resultado_zerado.items()}
tamanho_reducao = {vrp: len(nos) for vrp, nos in resultado_reducao.items()}

df_nos = pd.DataFrame({'NodeID': d.getNodeNameID()})
df_nos['ZP_zerado'] = ''
df_nos['ZP_reducao'] = ''

for node in df_nos['NodeID']:
    # Atribuição Zerado
    vrps_z = [v for v, nos in resultado_zerado.items() if node in nos]
    if vrps_z:
        df_nos.loc[df_nos['NodeID'] == node, 'ZP_zerado'] = min(vrps_z, key=lambda x: tamanho_zerado[x])

    # Atribuição Redução
    vrps_r = [v for v, nos in resultado_reducao.items() if node in nos]
    if vrps_r:
        df_nos.loc[df_nos['NodeID'] == node, 'ZP_reducao'] = min(vrps_r, key=lambda x: tamanho_reducao[x])

print("\n🏁 Processamento finalizado. Tabela df_nos atualizada.")

🚀 Rodando simulação base para identificar Montante/Jusante...
➡️ Processando VRP: VRP.SSB.002
  🔗 Montante: PM.VRP.SSB.002 | Jusante: 7501
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.003
  🔗 Montante: PM.VRP.SSB.003 | Jusante: 1129
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.004
  🔗 Montante: PM.VRP.SSB.004 | Jusante: 8082
  ✅ Zerados: 85 | Redução: 73
➡️ Processando VRP: VRP.SSB.005_REATIVAR
  🔗 Montante: 5546 | Jusante: 5300
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.008
  🔗 Montante: 7838 | Jusante: PM.VRP.SSB.008
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.009
  🔗 Montante: PM.VRP.SSB.009 | Jusante: 1031
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.010
  🔗 Montante: PM.VRP.SSB.010 | Jusante: 7303
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.011
  🔗 Montante: PM.VRP.SSB.011 | Jusante: 6411
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.012A
  🔗 Montante: 155 | Jusante: 1


c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarni

  ✅ Zerados: 1590 | Redução: 1580
➡️ Processando VRP: VRP.SSB.012B
  🔗 Montante: PM.VRP.SSB.012 | Jusante: 6292
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.013
  🔗 Montante: PM.VRP.SSB.013 | Jusante: 6948
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.014
  🔗 Montante: PM.VRP.SSB.014 | Jusante: 7332
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.015
  🔗 Montante: PM.VRP.SSB.015 | Jusante: 154
  ✅ Zerados: 515 | Redução: 509
➡️ Processando VRP: VRP.SSB.017_REATIVAR
  🔗 Montante: 6488 | Jusante: 6487
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.018_Reativar
  🔗 Montante: 236 | Jusante: 5716
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.NOVA.001
  🔗 Montante: 7018 | Jusante: 7540


c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())


  ✅ Zerados: 309 | Redução: 297
➡️ Processando VRP: VRP.SSB.NOVA.002
  🔗 Montante: 8094 | Jusante: 8095
  ✅ Zerados: 57 | Redução: 45
➡️ Processando VRP: VRP.SSB.NOVA.003
  🔗 Montante: 1311 | Jusante: 794
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.NOVA.004
  🔗 Montante: 5522 | Jusante: 4387
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.NOVA.005
  🔗 Montante: 6287 | Jusante: 6928
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.NOVA.006
  🔗 Montante: 6537 | Jusante: 6536
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.NOVA.008
  🔗 Montante: 5137 | Jusante: 238
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.NOVA.009
  🔗 Montante: 1286 | Jusante: 3773
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.NOVA.010
  🔗 Montante: 619 | Jusante: 620
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP.SSB.NOVA.011
  🔗 Montante: 6562 | Jusante: 6564
  ✅ Zerados: 12 | Redução: 0
➡️ Processando VRP: VRP_FICTICIA
  🔗 Montante: 183 | Jusante: 

...

  ✅ Zerados: 192 | Redução: 180
➡️ Processando VRP: VRP_FICTICIA1
  🔗 Montante: 150 | Jusante: 151


c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarni

  ✅ Zerados: 329 | Redução: 317
➡️ Processando VRP: VRP_FICTICIA2
  🔗 Montante: 5434 | Jusante: 414141
  ✅ Zerados: 190 | Redução: 178
➡️ Processando VRP: VRP_FICTICIA_EBO
  🔗 Montante: 211 | Jusante: 342


c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: WARNING: System hydraulically unbalanced.
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarni

  ✅ Zerados: 571 | Redução: 559

🏁 Processamento finalizado. Tabela df_nos atualizada.


In [529]:
import pandas as pd

# -----------------------------------------------------------
# 1. GERAR CONTAGEM POR CRITÉRIO (ZERADO VS REDUÇÃO)
# -----------------------------------------------------------

# Conta nós atribuídos a cada VRP no método Zerado
count_z = df_nos[df_nos['ZP_zerado'] != '']['ZP_zerado'].value_counts()

# Conta nós atribuídos a cada VRP no método Redução
count_r = df_nos[df_nos['ZP_reducao'] != '']['ZP_reducao'].value_counts()

# -----------------------------------------------------------
# 2. UNIFICAR EM UMA TABELA COMPARATIVA
# -----------------------------------------------------------

comparativo = pd.concat([
    count_z.rename('Nos_Zerado'),
    count_r.rename('Nos_Reducao')
], axis=1).fillna(0)

# Formatação
comparativo = comparativo.reset_index().rename(columns={'index': 'VRP'})
comparativo['Nos_Zerado'] = comparativo['Nos_Zerado'].astype(int)
comparativo['Nos_Reducao'] = comparativo['Nos_Reducao'].astype(int)

# -----------------------------------------------------------
# 3. VALIDAÇÃO DE CONSISTÊNCIA
# -----------------------------------------------------------

# Critério: Redução deve cobrir pelo menos a mesma área que o Zerado
comparativo['Diferenca'] = comparativo['Nos_Reducao'] - comparativo['Nos_Zerado']
comparativo['Status'] = comparativo.apply(
    lambda x: '✅ OK' if x['Nos_Reducao'] >= x['Nos_Zerado'] else '⚠️ Revisar', axis=1
)

# -----------------------------------------------------------
# 4. ORDENAÇÃO E EXPORTAÇÃO
# -----------------------------------------------------------

# Ordenar por ordem alfabética da VRP conforme solicitado
comparativo = comparativo.sort_values(by='VRP', ascending=True).reset_index(drop=True)

# 5. VERIFICAR SE HÁ NÓS "CONFLITANTES" (Zerados em uma VRP mas Reduzidos em outra)
conflitos = df_nos[
    (df_nos['ZP_zerado'] != '') & 
    (df_nos['ZP_reducao'] != '') & 
    (df_nos['ZP_zerado'] != df_nos['ZP_reducao'])
]

# Exibir os resultados
print(f"Total de VRPs analisadas: {len(comparativo)}")
print(f"Nós com divergência de zona (Zera em uma, Reduz em outra): {len(conflitos)}")

# Salvar o comparativo para conferência externa
comparativo.to_csv('comparativo_final_vrps.csv', index=False)

comparativo

Total de VRPs analisadas: 10
Nós com divergência de zona (Zera em uma, Reduz em outra): 8


,VRP,Nos_Zerado,Nos_Reducao,Diferenca,Status
0,VRP.SSB.002,12,0,-12,⚠️ Revisar
1,VRP.SSB.004,73,73,0,✅ OK
2,VRP.SSB.012A,1578,1580,2,✅ OK
3,VRP.SSB.015,503,509,6,✅ OK
4,VRP.SSB.NOVA.001,179,179,0,✅ OK
5,VRP.SSB.NOVA.002,45,45,0,✅ OK
6,VRP_FICTICIA,180,180,0,✅ OK
7,VRP_FICTICIA1,317,317,0,✅ OK
8,VRP_FICTICIA2,178,178,0,✅ OK
9,VRP_FICTICIA_EBO,559,559,0,✅ OK


🚀 Rodando simulação base (Snapshot)...
⚙️ Processando 29 válvulas...
Analisando: VRP.SSB.002 | ✅ Z: 556 / R: 780
Analisando: VRP.SSB.003 | ✅ Z: 117 / R: 98
Analisando: VRP.SSB.004 | ✅ Z: 91 / R: 72
Analisando: VRP.SSB.005_REATIVAR | ✅ Z: 179 / R: 160
Analisando: VRP.SSB.008 | ✅ Z: 348 / R: 329
Analisando: VRP.SSB.009 | ✅ Z: 279 / R: 266
Analisando: VRP.SSB.010 | ✅ Z: 304 / R: 290
Analisando: VRP.SSB.011 | ✅ Z: 547 / R: 531
Analisando: VRP.SSB.012A | ✅ Z: 1591 / R: 1579
Analisando: VRP.SSB.012B | ✅ Z: 1591 / R: 1579
Analisando: VRP.SSB.013 | ✅ Z: 175 / R: 533
Analisando: VRP.SSB.014 | ✅ Z: 779 / R: 760
Analisando: VRP.SSB.015 | ✅ Z: 523 / R: 510
Analisando: VRP.SSB.017_REATIVAR | ✅ Z: 124 / R: 105
Analisando: VRP.SSB.018_Reativar | ✅ Z: 175 / R: 162
Analisando: VRP.SSB.NOVA.001 | ✅ Z: 19 / R: 0
Analisando: VRP.SSB.NOVA.002 | ✅ Z: 19 / R: 0
Analisando: VRP.SSB.NOVA.003 | ✅ Z: 40 / R: 31
Analisando: VRP.SSB.NOVA.004 | ✅ Z: 19 / R: 0
Analisando: VRP.SSB.NOVA.005 | ✅ Z: 372 / R: 355
Analisa

,VRP,Nos_Zerado,Nos_Reducao,Status
0,VRP.SSB.002,556,780,✅ OK
1,VRP.SSB.003,117,98,⚠️ Revisar
2,VRP.SSB.004,91,72,⚠️ Revisar
3,VRP.SSB.005_REATIVAR,179,160,⚠️ Revisar
4,VRP.SSB.008,348,329,⚠️ Revisar
5,VRP.SSB.009,279,266,⚠️ Revisar
6,VRP.SSB.010,304,290,⚠️ Revisar
7,VRP.SSB.011,547,531,⚠️ Revisar
8,VRP.SSB.012A,1591,1579,⚠️ Revisar
9,VRP.SSB.012B,1591,1579,⚠️ Revisar


In [ ]:
print(gdf_nos.columns)
print(df_reducao.columns)
print(df_zerado.columns)

Index(['ID', 'ELEVATION', 'DEMAND', 'PATTERN', 'X', 'Y', 'geometry', 'P_00',
       'P_01', 'P_02', 'P_03', 'P_04', 'P_05', 'P_06', 'P_07', 'P_08', 'P_09',
       'P_10', 'P_11', 'P_12', 'P_13', 'P_14', 'P_15', 'P_16', 'P_17', 'P_18',
       'P_19', 'P_20', 'P_21', 'P_22', 'P_23', 'P_24'],
      dtype='object')
Index(['NodeID', 'ZP_reducao'], dtype='object')
Index(['NodeID', 'ZP_zerado'], dtype='object')


In [335]:
gdf_nos['ID'] = gdf_nos['ID'].astype(str)
df_reducao['NodeID'] = df_reducao['NodeID'].astype(str)
df_zerado['NodeID'] = df_zerado['NodeID'].astype(str)

# merge redução
gdf_nos = gdf_nos.merge(
    df_reducao,
    left_on='ID',
    right_on='NodeID',
    how='left'
)

# remover coluna duplicada
gdf_nos = gdf_nos.drop(columns='NodeID')

# merge zerado
gdf_nos = gdf_nos.merge(
    df_zerado,
    left_on='ID',
    right_on='NodeID',
    how='left'
)

# remover coluna duplicada
gdf_nos = gdf_nos.drop(columns='NodeID')

In [336]:
gdf_nos

,ID,ELEVATION,DEMAND,PATTERN,X,Y,geometry,P_00,P_01,P_02,...,P_17,P_18,P_19,P_20,P_21,P_22,P_23,P_24,ZP_reducao,ZP_zerado
0,0,1026.20,0.093873,VZ1.SAT.NBN.014,184364.028,8245057.145,POINT (184364.028 8245057.145),20.752228,20.825541,20.861477,...,20.430929,20.413698,20.369646,20.422388,20.472906,20.567282,20.639090,20.752213,VRP.CND.002,VRP.CND.002
1,1,1026.20,0.077392,VZ1.SAT.NBN.014,184365.120,8245055.470,POINT (184365.12 8245055.47),20.750801,20.824574,20.860735,...,20.427492,20.410152,20.365824,20.418896,20.469730,20.564697,20.636953,20.750786,VRP.CND.002,VRP.CND.002
2,2,1026.20,0.038697,VZ1.SAT.NBN.014,184365.848,8245054.355,POINT (184365.848 8245054.355),20.750063,20.824072,20.860350,...,20.425711,20.408316,20.363844,20.417088,20.468086,20.563358,20.635849,20.750048,VRP.CND.002,VRP.CND.002
3,3,1035.92,0.000000,VZ1.SAT.NBN.014,184122.502,8245560.306,POINT (184122.502 8245560.306),12.252913,12.309916,12.337867,...,12.003163,11.989783,11.955503,11.996499,12.035783,12.109170,12.164978,12.252922,VRP.CND.003,VRP.CND.003
4,4,1036.34,0.000000,VZ1.SAT.NBN.014,184125.377,8245557.215,POINT (184125.377 8245557.215),11.834476,11.890976,11.918680,...,11.586930,11.573668,11.539690,11.580324,11.619262,11.692001,11.747317,11.834484,VRP.CND.003,VRP.CND.003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2867,1913,1040.39,0.000000,;,181359.322,8242746.069,POINT (181359.322 8242746.069),10.000000,10.000000,10.000000,...,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,VRP.NBN.NOVA.004,VRP.NBN.NOVA.004
2868,1950,1041.84,0.000000,;,181630.241,8242894.674,POINT (181630.241 8242894.674),9.999335,9.999515,9.999607,...,9.998580,9.998535,9.998465,9.998534,9.998692,9.998861,9.999094,9.999335,VRP.NBN.NOVA.005,VRP.NBN.NOVA.005
2869,1961,1041.84,0.000000,;,181631.353,8242893.094,POINT (181631.353 8242893.094),10.000000,10.000000,10.000000,...,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,VRP.NBN.NOVA.005,VRP.NBN.NOVA.005
2870,1962,1046.30,0.000000,;,181602.747,8243099.993,POINT (181602.747 8243099.993),34.655254,37.092739,38.351780,...,24.078537,23.751993,22.800869,23.753550,25.888121,28.196152,31.371529,34.655243,VRP.NBN.NOVA.006,VRP.SPW.010


🚀 Rodando simulação base (Snapshot)...
⚙️ Processando 10 válvulas...
Analisando: VRP.CND.001 | ✅ Z: 859 / R: 858
Analisando: VRP.CND.002 | ✅ Z: 119 / R: 118
Analisando: VRP.CND.003 | ✅ Z: 126 / R: 125
Analisando: VRP.CND.004 | ✅ Z: 123 / R: 122
Analisando: VRP.GUA.014 | ✅ Z: 181 / R: 180
Analisando: VRP.NBN.001 | ✅ Z: 118 / R: 117
Analisando: VRP.NBN.002 | ✅ Z: 90 / R: 89
Analisando: VRP.NBN.003 | ✅ Z: 78 / R: 77
Analisando: VRP.NBN.005 | ✅ Z: 206 / R: 205
Analisando: VRP.SPW.010 | ✅ Z: 2854 / R: 2853

🏁 Processamento Finalizado!


,VRP,Nos_Zerado,Nos_Reducao,Status
0,VRP.CND.001,859,858,⚠️ Revisar
1,VRP.CND.002,119,118,⚠️ Revisar
2,VRP.CND.003,126,125,⚠️ Revisar
3,VRP.CND.004,123,122,⚠️ Revisar
4,VRP.GUA.014,181,180,⚠️ Revisar
5,VRP.NBN.001,118,117,⚠️ Revisar
6,VRP.NBN.002,90,89,⚠️ Revisar
7,VRP.NBN.003,78,77,⚠️ Revisar
8,VRP.NBN.005,206,205,⚠️ Revisar
9,VRP.SPW.010,2854,2853,⚠️ Revisar


In [123]:
gdf_nos['ZP'].unique()

array([''], dtype=object)

In [90]:
gdf_nos.to_file('Shape\\NBN CND\\nos_pressao_24h.shp')